# COVID-19 Exploratory Data Analysis: Data Preparation

**Goal:** Turn five related COVID-19 excel tables (cases and deaths, location, hospitalization, vaccination, testing) into one clean, analysis-ready dataset.

**What this notebook does**

1. Set up the project and load the five Excel sheets
2. Profile data quality (types, missing values, duplicates)
3. Standardize join keys 
4. Validate and merge the tables into one dataset
5. Save the result as Parquet for the analysis notebooks and future use

**Data source:** Our World in Data COVID-19 dataset, split into five tables. 

#### 1.Importing These Libraries for My Exploratory Phase


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

# Project paths
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully!")
print("Project directory:", PROJECT_DIR)

Libraries imported successfully!
Project directory: C:\Users\Keith K\Downloads\Covid Portfolio


#### 2. Defining and Initializing the Project's Directory Structure and Connecting to My Data

In [2]:

from pathlib import Path

# Main project directory
PROJECT_DIR = Path(
    r"C:\Users\Keith K\Downloads\Covid Portfolio\COVID19_Data_Analysis"
)

# Project subfolders
DATA_DIR = PROJECT_DIR / "data"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"

# Create output folders if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)
print("Figures directory:", FIGURES_DIR)

Project directory: C:\Users\Keith K\Downloads\Covid Portfolio\COVID19_Data_Analysis
Data directory: C:\Users\Keith K\Downloads\Covid Portfolio\COVID19_Data_Analysis\data
Output directory: C:\Users\Keith K\Downloads\Covid Portfolio\COVID19_Data_Analysis\outputs
Figures directory: C:\Users\Keith K\Downloads\Covid Portfolio\COVID19_Data_Analysis\outputs\figures


#### 3.Connecting Excel workbooks to Jupyter notebooks 

In [3]:
file_path = PROJECT_DIR / "Covid19_Data.xlsx"

deaths = pd.read_excel(file_path, sheet_name="CovidDeaths")
location = pd.read_excel(file_path, sheet_name="Location")
hospitalization = pd.read_excel(file_path, sheet_name="Hospitalization")
vaccination = pd.read_excel(file_path, sheet_name="Vaccination")
testing = pd.read_excel(file_path, sheet_name="Testing")

print("All five tables loaded successfully!")

All five tables loaded successfully!


#### 4.Loading and previewing the dataset

In [4]:
tables = {
    "CovidDeaths": deaths,
    "Location": location,
    "Hospitalization": hospitalization,
    "Vaccination": vaccination,
    "Testing": testing
}

for name, df in tables.items():
    print(f"\n{name}")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    display(df.head(5))


CovidDeaths
Rows: 562504
Columns: 11


,code,country,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,stringency_index,reproduction_rate
0,AFG,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,AFG,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2,AFG,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
3,AFG,Afghanistan,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,NaN
4,AFG,Afghanistan,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,NaN



Location
Rows: 562504
Columns: 11


,code,continent,country,date,population,population_density,median_age,gdp_per_capita,extreme_poverty,diabetes_prevalence,handwashing_facilities
0,AFG,Asia,Afghanistan,2020-01-01,40578847.0,62.215549,16.752001,1964.452271,NaN,11.7,51.938343
1,AFG,Asia,Afghanistan,2020-01-02,40578847.0,62.215549,16.752001,1964.452271,NaN,11.7,51.938343
2,AFG,Asia,Afghanistan,2020-01-03,40578847.0,62.215549,16.752001,1964.452271,NaN,11.7,51.938343
3,AFG,Asia,Afghanistan,2020-01-04,40578847.0,62.215549,16.752001,1964.452271,NaN,11.7,51.938343
4,AFG,Asia,Afghanistan,2020-01-05,40578847.0,62.215549,16.752001,1964.452271,NaN,11.7,51.938343



Hospitalization
Rows: 562504
Columns: 7


,code,country,date,hosp_patients,weekly_hosp_admissions,icu_patients,weekly_icu_admissions
0,AFG,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,2020-01-04,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN



Vaccination
Rows: 562504
Columns: 10


,code,country,date,total_vaccinations,people_vaccinated,people_fully_vaccinated,total_boosters,new_vaccinations,new_vaccinations_smoothed,new_people_vaccinated_smoothed
0,AFG,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Testing
Rows: 562504
Columns: 8


,code,country,date,total_tests,new_tests,new_tests_smoothed,positive_rate,tests_per_case
0,AFG,Afghanistan,2020-01-01,NaN,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,2020-01-02,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,2020-01-03,NaN,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,2020-01-04,NaN,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,2020-01-05,NaN,NaN,NaN,NaN,NaN


## Data cleaning the dataset

#### 1. Standardizing the column names 

In [5]:
for name, df in tables.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

print("Column names standardized.")

Column names standardized.


#### 4. Checking the data types 

In [6]:
for name, df in tables.items():
    print(f"\n{name}")
    print(df.dtypes)


CovidDeaths
code                              str
country                           str
date                   datetime64[us]
total_cases                   float64
new_cases                     float64
new_cases_smoothed            float64
total_deaths                  float64
new_deaths                    float64
new_deaths_smoothed           float64
stringency_index              float64
reproduction_rate             float64
dtype: object

Location
code                                 str
continent                            str
country                              str
date                      datetime64[us]
population                       float64
population_density               float64
median_age                       float64
gdp_per_capita                   float64
extreme_poverty                  float64
diabetes_prevalence              float64
handwashing_facilities           float64
dtype: object

Hospitalization
code                                 str
country               

#### 4.Checking for missing values 
Many columns are sparse by nature: not every country reports has hospital or testing data. I'll leave these as `NaN` rather than filling them, because a filled value would be invented data. Each analysis will filter to the countries and periods that actually report the metric.

In [7]:
for name, df in tables.items():
    print(f"\n{name} missing values")
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    display(missing.to_frame("Missing Count"))


CovidDeaths missing values


,Missing Count
reproduction_rate,378769
stringency_index,359881
new_cases_smoothed,11645
new_cases,10526
new_deaths_smoothed,10476
new_deaths,9354
total_deaths,5998
total_cases,5998



Location missing values


,Missing Count
handwashing_facilities,276752
extreme_poverty,175039
gdp_per_capita,89852
diabetes_prevalence,65938
population_density,11996
median_age,2437
population,2437



Hospitalization missing values


,Missing Count
weekly_icu_admissions,551511
weekly_hosp_admissions,542838
icu_patients,527679
hosp_patients,527062



Vaccination missing values


,Missing Count
total_boosters,519920
new_vaccinations,509828
people_fully_vaccinated,501693
people_vaccinated,499693
total_vaccinations,495482
new_vaccinations_smoothed,375864
new_people_vaccinated_smoothed,375864



Testing missing values


,Missing Count
new_tests,487133
total_tests,483149
tests_per_case,461873
positive_rate,460971
new_tests_smoothed,458566


#### 5.Checking for duplicates 

In [8]:
for name, df in tables.items():
    print(f"\n{name}")
    print("Duplicate full rows:", df.duplicated().sum())
    print(
        "Duplicate country-date keys:",
        df.duplicated(subset=["code", "date"]).sum()
    )


CovidDeaths
Duplicate full rows: 0
Duplicate country-date keys: 0

Location
Duplicate full rows: 0
Duplicate country-date keys: 0

Hospitalization
Duplicate full rows: 0
Duplicate country-date keys: 0

Vaccination
Duplicate full rows: 0
Duplicate country-date keys: 0

Testing
Duplicate full rows: 0
Duplicate country-date keys: 0


#### 6.Standardizing country codes and date values for joining tables 

In [9]:
for df in tables.values():
    df["code"] = df["code"].astype("string").str.strip()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    
for name, df in tables.items():
    print(
        name,
        "Missing codes:", df["code"].isna().sum(),
        "Missing dates:", df["date"].isna().sum()
    )

CovidDeaths Missing codes: 0 Missing dates: 0
Location Missing codes: 0 Missing dates: 0
Hospitalization Missing codes: 0 Missing dates: 0
Vaccination Missing codes: 0 Missing dates: 0
Testing Missing codes: 0 Missing dates: 0


##### 7.Checking the keys before joing the tables

In [10]:
for name, df in tables.items():
    duplicate_keys = df.duplicated(["code", "date"]).sum()
    print(f"{name}: {duplicate_keys} duplicate keys")

CovidDeaths: 0 duplicate keys
Location: 0 duplicate keys
Hospitalization: 0 duplicate keys
Vaccination: 0 duplicate keys
Testing: 0 duplicate keys


#### 8. Joining the tables Keeping useful columns for the analysis and avoiding  repeated country columns

In [11]:
location_cols = [
    "code", "date", "continent", "population",
    "population_density", "median_age", "gdp_per_capita",
    "extreme_poverty", "diabetes_prevalence",
    "handwashing_facilities"
]

hospital_cols = [
    "code", "date", "hosp_patients",
    "weekly_hosp_admissions", "icu_patients",
    "weekly_icu_admissions"
]

vaccination_cols = [
    "code", "date", "total_vaccinations",
    "people_vaccinated", "people_fully_vaccinated",
    "total_boosters", "new_vaccinations",
    "new_vaccinations_smoothed",
    "new_people_vaccinated_smoothed"
]

testing_cols = [
    "code", "date", "total_tests", "new_tests",
    "new_tests_smoothed", "positive_rate", "tests_per_case"
]

df = deaths.merge(
    location[location_cols],
    on=["code", "date"],
    how="outer",
    validate="one_to_one"
)

df = df.merge(
    hospitalization[hospital_cols],
    on=["code", "date"],
    how="outer",
    validate="one_to_one"
)

df = df.merge(
    vaccination[vaccination_cols],
    on=["code", "date"],
    how="outer",
    validate="one_to_one"
)

df = df.merge(
    testing[testing_cols],
    on=["code", "date"],
    how="outer",
    validate="one_to_one"
)

print("Merged data shape:", df.shape)
display(df.head())

Merged data shape: (562504, 35)


,code,country,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,stringency_index,...,people_fully_vaccinated,total_boosters,new_vaccinations,new_vaccinations_smoothed,new_people_vaccinated_smoothed,total_tests,new_tests,new_tests_smoothed,positive_rate,tests_per_case
0,ABW,Aruba,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,Aruba,2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,Aruba,2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,Aruba,2020-01-04,0.0,0.0,NaN,0.0,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,Aruba,2020-01-05,0.0,0.0,NaN,0.0,0.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
print("Rows and columns:", df.shape)
print("Countries:", df["code"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

display(df.isnull().sum().sort_values(ascending=False).head(20))

Rows and columns: (562504, 35)
Countries: 239
Date range: 2020-01-01 00:00:00 to 2026-07-19 00:00:00


weekly_icu_admissions             551511
weekly_hosp_admissions            542838
icu_patients                      527679
hosp_patients                     527062
total_boosters                    519920
new_vaccinations                  509828
people_fully_vaccinated           501693
people_vaccinated                 499693
total_vaccinations                495482
new_tests                         487133
total_tests                       483149
tests_per_case                    461873
positive_rate                     460971
new_tests_smoothed                458566
reproduction_rate                 378769
new_vaccinations_smoothed         375864
new_people_vaccinated_smoothed    375864
stringency_index                  359881
handwashing_facilities            276752
extreme_poverty                   175039
dtype: int64

#### 4. Convert it a parquet file for analysis and easier use

In [12]:
df.to_parquet("covid_merged.parquet", index=False)

print("Parquet file saved successfully!")

Parquet file saved successfully!
